In [1]:
import sys, re, json
sys.path.append('..') 

from scripts.constants import *
from scripts.utils import *
from scripts.sedona_config import *
from scripts.utils import translate_tile_name

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

In [2]:
# IN paths
vom_dir = RASTER_IN_DIR / "Defra" / "VOM"
vom_lad_dir = vom_dir / "LADs"
chm_lad_tiles_path = vom_lad_dir / "LAD_CHM_tiles_paths.json"
T3_30_300_DIR = VECTOR_OUT_DIR / "3-30-300"
T3_dir = T3_30_300_DIR / "T3"
# T30_dir = T3_30_300_DIR / "T30"
# T300_dir = T3_30_300_DIR / "T300"
trees_dir = T3_30_300_DIR / "VOM_Trees"
chm_lad_tiles_dict = json.load(open(chm_lad_tiles_path))
imd_lsoa_bua_boundaries_path = VECTOR_OUT_DIR / "IMD" / "English_IMD_2019_BUA_filtered_boundaries.geojson"
# buildings_path = VECTOR_IN_DIR / "EDINA" / "Buildings_6183" / "edition_17_0_new_format.gpkg"
buildings_path = VECTOR_IN_DIR / "EDINA" / "Buildings_6183" / "Buildings_6183.parquet"

In [3]:
project_crs = 'EPSG:27700'
os_5km_boundaries_path = VECTOR_IN_DIR / "OS" / "National_Grid" / "5km_grid_region.shp"
os_5km_boundaries_gdf = gpd.read_file(os_5km_boundaries_path).to_crs(project_crs)

In [4]:
geo_level = 'LSOA11CD'
geo_code = 'E01000107'
buffer_size = 100
imd_lsoa_bua_gdf = gpd.read_file(imd_lsoa_bua_boundaries_path)
imd_lsoa_bua_buffer_gdf = imd_lsoa_bua_gdf.copy()
imd_lsoa_bua_buffer_gdf['geometry'] = imd_lsoa_bua_buffer_gdf['geometry'].buffer(buffer_size)
geo_boundary_gdf = imd_lsoa_bua_gdf[imd_lsoa_bua_gdf[geo_level] == geo_code].dissolve()[['geometry', geo_level]]

In [74]:
imd_lsoa_bua_gdf[imd_lsoa_bua_gdf[geo_level] == geo_code]

,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,LSOA21NMW,LAD22CD,LAD22NM,LAD22NMW,BUA22CD,BUA22NMW,BUA22NMG,RGN22NMW,BUA22NM,RGN22CD,RGN22NM,geometry
103,E01000107,Barking and Dagenham 010C,E01000107,Barking and Dagenham 010C,None,E09000002,Barking and Dagenham,None,E63004859,None,None,None,Barking and Dagenham,E12000007,London,"POLYGON ((551242.439 184701.751, 551202.165 18..."


In [80]:
def get_overlapping_tiles(imd_lsoa_bua_buffer_gdf, os_5km_boundaries_gdf, geo_level, geo_code):
    # Select one feature from imd_lsoa_bua_buffer_gdf
    selected_feature = imd_lsoa_bua_buffer_gdf[imd_lsoa_bua_buffer_gdf[geo_level] == geo_code]

    # Get the overlapping features
    overlapping_tiles_lst = gpd.overlay(selected_feature, os_5km_boundaries_gdf, how='intersection')['TILE_NAME'].unique().tolist()

    return overlapping_tiles_lst

In [84]:
overlapping_tiles_lst = get_overlapping_tiles(imd_lsoa_bua_buffer_gdf, os_5km_boundaries_gdf, geo_level, geo_code)
overlapping_tiles_lst

['TQ58NW', 'TQ58SW']

In [133]:
get_vom_trees_paths(overlapping_tiles_lst, vom_tree_pair_dict)

['/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TQ5085_2019.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TQ5085_2019.gpkg']

In [108]:
vom_tree_pair_dict['E09000002']

[('/maps/acz25/phd-thesis-data/input/raster/Defra/VOM/unzipped_tiles/2020/VOM_TQ4585_P_12151_20201212_20201212.tif',
  '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TQ4585_2020.gpkg'),
 ('/maps/acz25/phd-thesis-data/input/raster/Defra/VOM/unzipped_tiles/2020/VOM_TQ4085_P_12151_20201212_20201212.tif',
  '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TQ4085_2020.gpkg'),
 ('/maps/acz25/phd-thesis-data/input/raster/Defra/VOM/unzipped_tiles/2020/VOM_TQ4580_P_12151_20201212_20201212.tif',
  '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TQ4580_2020.gpkg'),
 ('/maps/acz25/phd-thesis-data/input/raster/Defra/VOM/unzipped_tiles/2020/VOM_TQ4080_P_12151_20201212_20201212.tif',
  '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TQ4080_2020.gpkg'),
 ('/maps/acz25/phd-thesis-data/input/raster/Defra/VOM/unzipped_tiles/2020/VOM_TQ4590_P_12151_20201212_20201212.tif',
  '/maps/acz25/phd-thesis-data/output/v

In [ ]:
vom_trees_path_lst = [v[0][1] for _,v in vom_tree_pair_dict.items() if v[0][1] is not None]
for tile_name in overlapping_tiles_lst:
    
    translated_tile_name = translate_tile_name(tile_name).upper()
    print(tile_name, translated_tile_name)
    trees_path_lst = set([path for path in vom_trees_path_lst if translated_tile_name in path])

TQ58NW TQ5085
TQ58SW TQ5080


In [131]:
vom_trees_path_lst

['/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ3525_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4525_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4025_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4535_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4035_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4530_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4030_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ5025_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ5035_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ5030_2021.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_NZ4515_

In [ ]:
vom_trees_path_lst = [path_pair[1] for _,v in vom_tree_pair_dict.items() for path_pair in v if path_pair[1] is not None]
len(vom_trees_path_lst)

8586

In [132]:
def get_vom_trees_paths(overlapping_tiles_lst: list, vom_tree_pair_dict: dict) -> list:

    vom_trees_path_lst = [path_pair[1] for _,v in vom_tree_pair_dict.items() for path_pair in v if path_pair[1] is not None]
    for tile_name in overlapping_tiles_lst:
        translated_tile_name = translate_tile_name(tile_name).upper()
        trees_path_lst = [path for path in vom_trees_path_lst if translated_tile_name in path]

        return trees_path_lst


In [5]:
def extract_grid_reference(filename: str) -> str|None:
    """
    Extracts a grid reference from a given filename.
    The function searches for a pattern in the filename that matches 'VOM' or 'VOM_HS'
    followed by an underscore, a two-letter code, a four-digit number, and another underscore.
    If such a pattern is found, it returns the grid reference (the two-letter code and the four-digit number).
    If no match is found, it returns None.
    Parameters:
        filename (str | Path): The name of the file from which to extract the grid reference.
    Returns:
        str | None: The extracted grid reference if a match is found, otherwise None.
    """

    match = re.search(r'VOM_([A-Z]{2}\d{4})_', filename)
    if match:
        return match.group(1)
    return None

In [6]:
def check_tree_vom_pair(chm_path: str|Path, trees_dir: str|Path) -> bool:
    """
    Check if a CHM file has a corresponding tree file.
    Parameters:
        chm_path (str | Path): The path to the CHM file.
        trees_dir (str | Path): The directory containing the tree files.
    Returns:
        bool: True if a corresponding tree file exists, otherwise False.
    """

    tile_name = extract_grid_reference(chm_path)
    chm_path = chm_path if isinstance(chm_path, Path) else Path(chm_path)
    year = chm_path.parent.name

    trees_path = trees_dir / f"VOM_trees_{tile_name}_{year}.gpkg"

    if trees_path.exists():
        return str(trees_path)

In [42]:
vom_tree_pair_dict = {geo_code: [(chm_path, check_tree_vom_pair(chm_path, trees_dir)) for chm_path in lst] for geo_code, lst in chm_lad_tiles_dict.items()}

In [43]:
# Convert vom_tree_pair_dict to a DataFrame
vom_tree_pair_df = pd.DataFrame.from_dict(vom_tree_pair_dict, orient='index').stack().reset_index()
vom_tree_pair_df.columns = ['geo_code', 'index', 'pair']
vom_tree_pair_df = vom_tree_pair_df.drop(columns=['index'])
vom_tree_pair_df[['chm_path', 'tree_path']] = pd.DataFrame(vom_tree_pair_df['pair'].tolist(), index=vom_tree_pair_df.index)
vom_tree_pair_df = vom_tree_pair_df.drop(columns=['pair'])
no_tree_file_df = vom_tree_pair_df[vom_tree_pair_df['tree_path'].isnull()]

In [9]:
trees_path_lst = [tree_path for _, tree_path in vom_tree_pair_dict[geo_code] if tree_path is not None]
trees_path_lst

['/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TL4555_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TL4055_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TL4550_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TL4050_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TL4560_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_TL4060_2023.gpkg']

In [10]:
trees_gdf_lst = [gpd.read_file(tree_path) for tree_path in trees_path_lst]

In [11]:
merged_trees_gdf = gpd.GeoDataFrame(pd.concat(trees_gdf_lst, ignore_index=True))
geo_trees_gdf = merged_trees_gdf[merged_trees_gdf.area > 4].reset_index(drop=True)
geo_trees_gdf['treeID'] = range(len(geo_trees_gdf))
geo_trees_gdf

,treeID,height,area,geometry
0,0,7.306001,26.005502,"MULTIPOLYGON (((545377 260000, 545377 259999, ..."
1,1,4.223001,11.002304,"MULTIPOLYGON (((545620 260000, 545620 259998, ..."
2,2,2.770000,7.001485,"MULTIPOLYGON (((546681 260000, 546681 259998, ..."
3,3,11.491000,50.011463,"MULTIPOLYGON (((547073 260000, 547073 259996, ..."
4,4,3.234000,11.002518,"MULTIPOLYGON (((547284 259999, 547284 259996, ..."
...,...,...,...,...
430594,430594,10.002003,46.010006,"MULTIPOLYGON (((544757 260007, 544757 260005, ..."
430595,430595,4.627001,33.007131,"MULTIPOLYGON (((544919 260006, 544919 260005, ..."
430596,430596,3.951002,39.009325,"MULTIPOLYGON (((543664 260011, 543664 260009, ..."
430597,430597,10.625002,30.007123,"MULTIPOLYGON (((543787 260005, 543787 260004, ..."


In [11]:
# from shapely.ops import unary_union
# dissolved_geometry = unary_union(merged_trees_gdf.geometry)

In [12]:
os.environ["JAVA_HOME"] = JAVA_HOME
sedona = get_spark()

24/12/09 19:26:21 WARN Utils: Your hostname, kinabalu resolves to a loopback address: 127.0.1.1; using 128.232.93.1 instead (on interface eno12399np0)
24/12/09 19:26:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
https://artifacts.unidata.ucar.edu/repository/unidata-all added as a remote repository with the name: repo-1
Ivy Default Cache set to: /home/acz25/.ivy2/cache
The jars for the packages stored in: /home/acz25/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
net.postgis#postgis-jdbc added as a dependency
net.postgis#postgis-geometry added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fd069969-fe5a-4b49-8d73-b85e840d9c63;1.0
	confs: [default]


:: loading settings :: url = jar:file:/maps-priv/maps/acz25/miniconda3/envs/3-30-300-env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.sedona#sedona-spark-3.5_2.12;1.7.0 in central
	found org.apache.sedona#sedona-common;1.7.0 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found net.sf.geographiclib#GeographicLib-Java;1.52 in central
	found com.github.ben-manes.caffeine#caffeine;2.9.2 in central
	found org.checkerframework#checker-qual;3.10.0 in central
	found com.google.error

In [13]:
boundaries_sdf = sedona.createDataFrame(imd_lsoa_bua_gdf.drop(columns=['LSOA21NMW', 'LAD22NMW', 'BUA22NMG', 'BUA22NMW', 'RGN22NMW'], axis=1))
boundaries_sdf.createOrReplaceTempView('boundaries')
buildings_sdf = sedona.read.format("geoparquet").load(str(buildings_path))
buildings_sdf.createOrReplaceTempView("buildings")
buildings_sdf.printSchema()
geo_trees_sdf = sedona.createDataFrame(geo_trees_gdf)
geo_trees_sdf.createOrReplaceTempView("geo_trees")

root
 |-- building_area: double (nullable = true)
 |-- distance_building: float (nullable = true)
 |-- distance_water: double (nullable = true)
 |-- height: double (nullable = true)
 |-- map_simple_use: string (nullable = true)
 |-- map_use: string (nullable = true)
 |-- premise_floor_count: string (nullable = true)
 |-- premise_type: string (nullable = true)
 |-- premise_use: string (nullable = true)
 |-- premise_year: double (nullable = true)
 |-- verisk_building_id: long (nullable = true)
 |-- verisk_premise_id: long (nullable = true)
 |-- geometry: geometry (nullable = true)



In [14]:
geo_boundary_sdf = sedona.sql(f"""
    SELECT ST_Union_Aggr(geometry) AS geometry
    FROM boundaries
    WHERE {geo_level} = '{geo_code}'
""")
geo_boundary_sdf.createOrReplaceTempView("geo_boundary")

In [15]:
geo_buildings_sdf = sedona.sql("SELECT b.* FROM buildings b, geo_boundary g WHERE ST_Intersects(b.geometry, g.geometry)")
geo_buildings_sdf.createOrReplaceTempView("geo_buildings")
# geo_buildings_sdf.show()

In [16]:
geo_buildings_buffer_sdf = sedona.sql("""
    SELECT ST_Buffer(b.geometry, 100) AS geometry, b.verisk_premise_id
    FROM geo_buildings b
""")
geo_buildings_buffer_sdf.createOrReplaceTempView("building_buffers")

In [18]:
from sedona.core.SpatialRDD import CircleRDD, PolygonRDD
from sedona.utils.adapter import Adapter
from sedona.core.enums import GridType, IndexType
from sedona.core.spatialOperator import JoinQuery

In [25]:
geo_buildings_rdd = Adapter.toSpatialRdd(geo_buildings_sdf, 'geometry')
geo_buildings_rdd.analyze()
geo_buildings_buffer_rdd = Adapter.toSpatialRdd(geo_buildings_buffer_sdf, 'geometry')
geo_buildings_buffer_rdd.analyze()

geo_trees_rdd = Adapter.toSpatialRdd(geo_trees_sdf, 'geometry')
geo_trees_rdd.analyze()
geo_buildings_buffer_rdd.spatialPartitioning(GridType.KDBTREE)
geo_buildings_buffer_rdd.buildIndex(IndexType.RTREE, True)
geo_trees_rdd.spatialPartitioning(geo_buildings_buffer_rdd.getPartitioner())

24/12/09 19:33:34 WARN TaskSetManager: Stage 15 contains a task of very large size (1088 KiB). The maximum recommended task size is 1000 KiB.


In [39]:
buildings_trees_join_rdd = JoinQuery.DistanceJoinQuery(geo_trees_rdd, geo_buildings_buffer_rdd, True, True)

Py4JError: An error occurred while calling z:org.apache.sedona.core.spatialOperator.JoinQuery.DistanceJoinQuery. Trace:
py4j.Py4JException: Method DistanceJoinQuery([class org.apache.sedona.core.spatialRDD.SpatialRDD, class org.apache.sedona.core.spatialRDD.SpatialRDD, class java.lang.Boolean, class java.lang.Boolean]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:342)
	at py4j.Gateway.invoke(Gateway.java:276)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.lang.Thread.run(Thread.java:750)



In [40]:
geo_buildings_buffer_rdd = CircleRDD(geo_buildings_rdd, 100)

Py4JError: An error occurred while calling None.org.apache.sedona.core.spatialRDD.CircleRDD. Trace:
py4j.Py4JException: Constructor org.apache.sedona.core.spatialRDD.CircleRDD([class org.apache.sedona.core.spatialRDD.SpatialRDD, class java.lang.Integer]) does not exist
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:180)
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:197)
	at py4j.Gateway.invoke(Gateway.java:237)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.lang.Thread.run(Thread.java:750)



In [21]:
# Count the number of trees within each buffer
trees_within_buffer = sedona.sql("""
    SELECT b.verisk_premise_id, COUNT(t.treeID) AS tree_count
    FROM building_buffers b
    LEFT JOIN geo_trees t
    ON ST_Intersects(b.geometry, t.geometry)
    GROUP BY b.verisk_premise_id
""")
trees_within_buffer.show()

24/12/09 15:51:15 WARN TaskSetManager: Stage 15 contains a task of very large size (1088 KiB). The maximum recommended task size is 1000 KiB.


+-----------------+----------+
|verisk_premise_id|tree_count|
+-----------------+----------+
|         17295964|       147|
|         14997144|       100|
|         21471107|        54|
|          9703740|       154|
|         18727252|       141|
|          1878505|       170|
|         19261227|       120|
|         19160428|        77|
|         15384185|       119|
|         30010616|       104|
|         17634936|       126|
|         10968299|       204|
|          6715494|       176|
|         18010623|       165|
|         20436994|        96|
|         18287780|       139|
|         20266485|       171|
|         11789453|       121|
|         20622279|        99|
|          5786001|       146|
+-----------------+----------+
only showing top 20 rows



In [ ]:
x = trees_within_buffer.toPandas()
# x.to_csv(T3_dir / f"T3_{geo_code}.csv", index=False)